# Observabilidad, Lineage y Automatización (Senior)

Objetivos:
- Definir métricas y SLOs de datos.
- Entender OpenLineage/Marquez y cómo emitir eventos de lineage.
- Integrar calidad/lineage en CI/CD de datos.

## 1. Métricas de Observabilidad

Capas y métricas:
- Ingestión: latencia, throughput, fallos, lag (stream).
- Transformación: éxito/fracaso jobs, duración, recursos.
- Datasets: frescura, conteos, nulls, drift de esquema.

SLOs típicos:
- Frescura (Gold diario listo antes de 07:00).
- Latencia P95 de eventos < 5s en alertas.
- % jobs exitosos > 99.5%.

In [ ]:
# Ejemplo: cálculo simple de frescura y %nulls
import pandas as pd, numpy as np
from datetime import datetime, timedelta
ahora = pd.Timestamp(datetime.utcnow())
last_update = ahora - timedelta(minutes=37)
frescura_min = (ahora - last_update).total_seconds()/60.0
print('Frescura (minutos):', round(frescura_min,1))

df = pd.DataFrame({'a':[1,2,None,4],'b':['x','y','z',None]})
pct_nulls = df.isna().mean()*100
print('Porcentaje de nulls (%):
', pct_nulls.round(1))

## 2. Lineage (OpenLineage)

Conceptos:
- Job: unidad de ejecución (nombre, namespace).
- Run: instancia (runId).
- Datasets: input/output con namespaces (p.ej., s3://bucket/path o bq:project.dataset.table).

Señalización: producir eventos JSON a un backend (Marquez) con estados: START, COMPLETE, FAIL.

In [ ]:
# Evento OpenLineage (ejemplo JSON)
import json, uuid, time
run_id = str(uuid.uuid4())
event = {
  'eventType': 'COMPLETE',
  'eventTime': pd.Timestamp.utcnow().isoformat(),
  'job': {'namespace':'retail-pipelines','name':'silver.ventas_diarias'},
  'run': {'runId': run_id},
  'inputs': [
    {'namespace':'s3','name':'s3://datalake/bronze/ventas/2025-11-07.parquet'}
  ],
  'outputs': [
    {'namespace':'delta','name':'/silver/ventas_diarias'}
  ],
  'producer': 'course://observability-example'
}
print(json.dumps(event, indent=2))

print('
Nota: En producción se POSTea a un backend OpenLineage/Marquez.')

## 3. CI/CD de Datos

Prácticas:
- Tests de datos (calidad) como paso obligatorio (gates).
- Validación de esquemas y contratos (eventos y tablas).
- Linting/format y revisión de cambios de SQL/infra (IaC).
- Despliegues con canarios/shadow pipelines; feature flags.

In [ ]:
# Simulación gate de calidad (fallar pipeline si un test falla)
def quality_gate(df):
    tests = {
        'not_null_a': lambda d: d['a'].notna().all(),
        'range_a': lambda d: (d['a'].fillna(0).between(0,1000)).all(),
        'nulls_b_pct<50': lambda d: d['b'].isna().mean() < 0.5,
    }
    results = {name: fn(df) for name, fn in tests.items()}
    print(results)
    if not all(results.values()):
        raise SystemExit('Quality gate FAILED')

try:
    quality_gate(df)
    print('Quality gate PASSED')
except SystemExit as e:
    print(e)

## 4. Alertas y Paneles

- Alertas por SLO incumplido (frescura/latencia/error-rate).
- Paneles por dominio con métricas clave, tendencias y incidentes.
- Definir ownership de alertas (on-call).

## 5. Ejercicio

Define un set de SLOs/alertas para el dominio de pedidos y un evento de lineage para el job de agregación mensual.

In [ ]:
plantilla = '''
# Solución (borrador)
SLOs: [frescura, latencia, disponibilidad]
Alertas: [umbrales + canales]
Lineage: [job/run/inputs/outputs]
CI/CD: [tests obligatorios + validación de esquema]
'''
print(plantilla)

## 6. Resumen

- Observabilidad requiere métricas claras y SLOs medibles.
- OpenLineage unifica eventos de lineage multi-plataforma.
- CI/CD de datos evita regresiones con gates de calidad y contratos.

## 7. Dataset de ejemplo y métricas automatizadas
En esta sección cargamos un dataset de ventas de `04-recursos/datasets/ejemplo_ventas.csv` para calcular frescura, nulls, validación de esquema y un pequeño agregado que podría alimentar un panel.

In [ ]:
# Cargar dataset de ejemplo y calcular métricas
import pandas as pd
from datetime import datetime, timezone
from pathlib import Path

# Ruta relativa desde este notebook
csv_path = Path('..') / '04-recursos' / 'datasets' / 'ejemplo_ventas.csv'
print('Cargando:', csv_path.resolve())

df_ventas = pd.read_csv(csv_path, parse_dates=['fecha','updated_at'])
print('Filas:', len(df_ventas), '| Columnas:', len(df_ventas.columns))

# 1) Frescura con base en updated_at (minutos)
ahora = pd.Timestamp(datetime.now(tz=timezone.utc))
max_update = df_ventas['updated_at'].max()
frescura_min = (ahora - max_update).total_seconds() / 60.0
print(f'Frescura desde última actualización (min): {frescura_min:.1f}')

# 2) Porcentaje de nulls por columna
pct_nulls = df_ventas.isna().mean().sort_values(ascending=False) * 100
print('\n% Nulls por columna (desc):')
print(pct_nulls.round(1))

# 3) Validación simple de esquema
expected_cols = {
    'fecha','pedido_id','cliente_id','pais','canal','categoria','cantidad','precio_unitario','total','updated_at'
}
actual_cols = set(df_ventas.columns)
missing = expected_cols - actual_cols
extra = actual_cols - expected_cols
print('\nEsquema esperado OK?', len(missing)==0 and len(extra)==0)
if missing:
    print('Faltantes:', missing)
if extra:
    print('Extras:', extra)

# 4) Agregado rápido para un panel (ventas por día y canal)
agregado = (
    df_ventas.groupby(['fecha','canal'], dropna=False)['total']
    .sum()
    .reset_index()
    .sort_values(['fecha','canal'])
)
print('\nVentas por día y canal:')
print(agregado)
